# 02 – Huấn luyện ma trận thực nghiệm E1–E8 trên PyTorch

Ma trận 8 thực nghiệm có kiểm soát nhằm trả lời toàn diện hai câu hỏi nghiên cứu:
- **RQ1 (Học chuyển giao & Đánh đổi kiến trúc):** Đối chứng Scratch (E1, E5) vs. Frozen Backbone (E2, E6) vs. Full Fine-tuning (E3, E7) trên ResNet-18 và EfficientNet-B0.
- **RQ2 (Xử lý mất cân bằng lớp & Tăng cường dữ liệu):** Đối chứng hàm mất mát chuẩn CE (E3, E7) vs. Class-Weighted CE (E4, E8).

Mỗi thực nghiệm được theo dõi tự động, ghi log JSONL, lưu checkpoint theo Validation Macro-F1 và đồng bộ lên W&B Online.

## Cell 1 – Thiết lập môi trường và cấu hình siêu tham số

In [ ]:
from pathlib import Path
import sys, os, traceback, platform
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import make_dataloaders, seed_everything
from src.model import build_resnet18, build_efficientnet_b0
from src.train import ExperimentLogger, fit, compute_class_weights

try:
    import wandb
    _WANDB_OK = True
except ImportError:
    wandb = None
    _WANDB_OK = False

# ── Cấu hình siêu tham số cố định ─────────────────────────────
SEED          = 42
BATCH_SIZE    = 32
EPOCHS        = 10
WEIGHT_DECAY  = 1e-4
MAX_ATTEMPTS  = 3
RUN_SEEDS     = [42, 43, 44, 45, 46]
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'plant-disease-deep-learning')
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', None)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata      = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
NUM_CLASSES   = metadata['label'].nunique()
CLASS_WEIGHTS = compute_class_weights(metadata)

print(f'Phần cứng thực thi : {DEVICE}')
print(f'PyTorch version     : {torch.__version__}')
print(f'Số lớp phân loại    : {NUM_CLASSES}')
print(f'Trọng số lớp (Min-Max): {CLASS_WEIGHTS.min():.4f} - {CLASS_WEIGHTS.max():.4f}')
print(f'W&B Project         : {WANDB_PROJECT} (Trạng thái: {"Sẵn sàng" if _WANDB_OK else "Thiếu gói wandb"})')

## Cell 2 – Định nghĩa ma trận 8 thực nghiệm (E1–E8)

In [ ]:
# Ma trận: (Tên, Bộ tạo mô hình, Augmentation, Learning Rate, Dùng Weighted Loss)
EXPERIMENTS = [
    ('E1_resnet18_scratch',     lambda: build_resnet18(NUM_CLASSES, pretrained=False, freeze_backbone=False), False, 1e-3, False),
    ('E2_resnet18_frozen',      lambda: build_resnet18(NUM_CLASSES, pretrained=True,  freeze_backbone=True),  True,  1e-3, False),
    ('E3_resnet18_fullft',      lambda: build_resnet18(NUM_CLASSES, pretrained=True,  freeze_backbone=False), True,  1e-4, False),
    ('E4_resnet18_fullft_wce',  lambda: build_resnet18(NUM_CLASSES, pretrained=True,  freeze_backbone=False), True,  1e-4, True),
    ('E5_effnetb0_scratch',     lambda: build_efficientnet_b0(NUM_CLASSES, pretrained=False, freeze_backbone=False), False, 1e-3, False),
    ('E6_effnetb0_frozen',      lambda: build_efficientnet_b0(NUM_CLASSES, pretrained=True,  freeze_backbone=True),  True,  1e-3, False),
    ('E7_effnetb0_fullft',      lambda: build_efficientnet_b0(NUM_CLASSES, pretrained=True,  freeze_backbone=False), True,  1e-4, False),
    ('E8_effnetb0_fullft_wce',  lambda: build_efficientnet_b0(NUM_CLASSES, pretrained=True,  freeze_backbone=False), True,  1e-4, True),
]

(PROJECT_ROOT / 'outputs').mkdir(exist_ok=True)
(PROJECT_ROOT / 'checkpoints').mkdir(exist_ok=True)
logger = ExperimentLogger(PROJECT_ROOT / 'outputs' / 'experiment_log.jsonl')
print(f'Đã khởi tạo ma trận {len(EXPERIMENTS)} thực nghiệm đối chứng.')

## Cell 3 – Vòng lặp huấn luyện chính có kiểm soát

In [ ]:
histories = {}
models = {}

for exp_name, factory, use_aug, lr, use_wce in EXPERIMENTS:
    for run_seed in RUN_SEEDS:
        run_id = f'{exp_name}_seed{run_seed}'
        ckpt_path = PROJECT_ROOT / 'checkpoints' / f'{run_id}.pt'

        for attempt in range(1, MAX_ATTEMPTS + 1):
            attempt_id = f'{run_id}_attempt{attempt}'
            print(f'\n========== {attempt_id} ==========')
            wandb_run = None
            run_success = False

            if _WANDB_OK:
                try:
                    wandb_run = wandb.init(
                        project  = WANDB_PROJECT,
                        entity   = WANDB_ENTITY,
                        name     = attempt_id,
                        group    = exp_name,
                        job_type = 'training',
                        reinit   = 'finish_previous',
                        tags     = [exp_name, f'seed-{run_seed}', f'attempt-{attempt}'],
                        config   = {
                            'experiment': exp_name,
                            'seed': run_seed,
                            'batch_size': BATCH_SIZE,
                            'epochs': EPOCHS,
                            'learning_rate': lr,
                            'augmentation': use_aug,
                            'weighted_loss': use_wce,
                            'num_classes': NUM_CLASSES,
                            'device': str(DEVICE),
                            'attempt': attempt,
                        },
                    )
                    wandb_run.define_metric('epoch')
                    wandb_run.define_metric('*', step_metric='epoch')
                except Exception as wb_err:
                    print(f'  [W&B] Khởi tạo thất bại: {wb_err}')
                    wandb_run = None

            try:
                seed_everything(run_seed)
                loaders = make_dataloaders(metadata, batch_size=BATCH_SIZE, augmentation=use_aug)
                weights_to_pass = CLASS_WEIGHTS if use_wce else None

                model, history = fit(
                    model           = factory(),
                    loaders         = loaders,
                    epochs          = EPOCHS,
                    device          = DEVICE,
                    learning_rate   = lr,
                    class_weights   = weights_to_pass,
                    checkpoint_path = ckpt_path,
                    logger          = logger,
                    run_id          = run_id,
                    experiment_name = exp_name,
                    wandb_run       = wandb_run,
                )

                history_df = pd.DataFrame(history)
                histories[run_id] = history_df
                models[run_id] = model.cpu()
                csv_path = PROJECT_ROOT / 'outputs' / f'{run_id}_history.csv'
                history_df.to_csv(csv_path, index=False)

                if wandb_run is not None:
                    wandb_run.summary['status'] = 'success'
                    wandb_run.summary['best_val_macro_f1'] = history_df['val_macro_f1'].max()
                    wandb_run.save(str(csv_path))

                run_success = True
                print(f'  ✓ Hoàn tất thành công: {run_id} (Best Val F1: {history_df["val_macro_f1"].max():.4f})')
                break

            except Exception as error:
                print(f'  ✗ Lỗi lần thử {attempt}/{MAX_ATTEMPTS}: {error}')
                logger.log(
                    event='run_attempt_failed', run_id=run_id, experiment=exp_name, attempt=attempt,
                    error_type=type(error).__name__, error_message=str(error), traceback=traceback.format_exc()
                )
                if wandb_run is not None:
                    wandb_run.summary['status'] = 'failed'
                    wandb_run.summary['error_type'] = type(error).__name__
                    wandb_run.summary['error_msg'] = str(error)[:500]

            finally:
                if wandb_run is not None:
                    wandb_run.finish(exit_code=0 if run_success else 1)

print('\nToàn bộ quá trình huấn luyện ma trận thực nghiệm E1–E8 đã hoàn tất!')